# LoRA: технические проверки и подготовка данных
Это не LR search и не обучение Food-101. Проверяем CUDA/PEFT на искусственных входах.

1. Runtime → Change runtime type → бесплатный GPU.
2. Загрузить colab-bootstrap.zip, затем установить отдельное окружение.
3. Продолжать с проверок. Kernel не перезапускаем и его пакеты не заменяем.
Версии bootstrap-кандидата становятся принятым профилем только после полной приёмки.


## Загрузка единой версии кода

In [ ]:
from google.colab import files
from pathlib import Path
import hashlib, io, json, zipfile, os
uploaded = files.upload()  # Выбрать только colab-bootstrap.zip.
if len(uploaded) != 1:
    raise RuntimeError("Нужен один архив colab-bootstrap.zip")
root = Path('/content/lora_project')
root.mkdir(exist_ok=True)  # Повторный запуск допускает ту же неизменённую версию.
with zipfile.ZipFile(io.BytesIO(next(iter(uploaded.values())))) as archive:
    for name in archive.namelist():
        if not (root / name).resolve().is_relative_to(root.resolve()):
            raise RuntimeError("Небезопасный путь в архиве")
    incoming = json.loads(archive.read('SOURCE_MANIFEST.json'))
    expected_release = hashlib.sha256(json.dumps(incoming['files'], sort_keys=True).encode()).hexdigest()
    if expected_release != incoming['release_hash']:
        raise RuntimeError('Неверный release hash')
    for name, expected in incoming['files'].items():
        target = root / name
        if target.exists() and hashlib.sha256(target.read_bytes()).hexdigest() != expected:
            raise RuntimeError(f'В {target} другая версия; используйте новую папку проекта')
    archive.extractall(root)
manifest = json.loads((root / 'SOURCE_MANIFEST.json').read_text())
for name, expected in manifest['files'].items():
    actual = hashlib.sha256((root / name).read_bytes()).hexdigest()
    if actual != expected:
        raise RuntimeError(f"Hash mismatch: {name}")
os.chdir(root)
import sys
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
os.environ['HF_HOME'] = '/content/hf-cache'
print('Source verified:', manifest['release_hash'])


## Закреплённое окружение в отдельном venv
Пакеты notebook kernel не заменяются; перезапуск kernel не нужен.

In [ ]:
import subprocess, sys
from pathlib import Path
ENV = Path('/content/lora_env')
PYTHON = str(ENV / 'bin/python')
subprocess.run([sys.executable, '-m', 'venv', '--without-pip', str(ENV)], check=True)
# Host pip can manage a target interpreter without ensurepip inside that interpreter.
subprocess.run([sys.executable, '-m', 'pip', '--python', PYTHON, 'install',
                'torch==2.8.0', 'torchvision==0.23.0', '--index-url',
                'https://download.pytorch.org/whl/cu126'], check=True)
subprocess.run([sys.executable, '-m', 'pip', '--python', PYTHON, 'install',
                '-r', str(root/'requirements.txt')], check=True)
print('PINNED_ENV_READY', PYTHON)


## Проверка CUDA и постоянной папки результатов

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# Для общего хранилища укажите реальный путь выданной вам папки/ярлыка.
PERSISTENT_ROOT = Path('/content/drive/MyDrive/LoRA_experiment')
EXECUTOR_ID = 'participant_1'  # У каждого участника своё значение.
from datetime import datetime, timezone
SESSION_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
out = PERSISTENT_ROOT / 'executors' / EXECUTOR_ID / 'technical' / SESSION_ID
out.mkdir(parents=True, exist_ok=False)
import subprocess, sys
subprocess.run([PYTHON, 'bootstrap.py', '--require-cuda',
                '--storage', str(out), '--output', str(out / 'environment.json')], check=True)
subprocess.run([PYTHON, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], check=True)


## Реальная структура ViT и PEFT, без скачивания весов
Случайные веса, четыре метода, искусственные изображения 224×224. Проверяются модули, параметры, gradients и frozen weights. Это ещё **не замер вместимости microbatch16**.

In [ ]:
subprocess.run([PYTHON, 'test_params.py', '--device', 'cuda', '--image-size', '224',
                '--output', str(out / 'synthetic_smoke.json')], check=True)


## Замер начального training microbatch для Both
Три обновления по 32 порции из 16 изображений. При OOM сначала завершаем процесс, затем повторяем с 8/64, далее 4/128. Другие ошибки не маскируются под OOM. Validation, Trainer/resume и реальные данные проверяются на следующем этапе; этот замер не является полной приёмкой.
AMP может пропускать первые попытки при калибровке scale. Требуем три успешных обновления; пропуски и scale сохраняются в JSON. Десять пропусков подряд останавливают проверку.


In [ ]:
BATCH, ACCUMULATION = 16, 32
subprocess.run([PYTHON, 'test_params.py', '--config', 'B-iso', '--device', 'cuda',
                '--image-size', '224', '--batch-size', str(BATCH),
                '--accumulation', str(ACCUMULATION),
                '--output', str(out / f'profile_b{BATCH}_a{ACCUMULATION}.json')], check=True)


## Необязательно: загрузка исходных pretrained weights
Разрешён только полный commit SHA. Его один раз разрешаем и сохраняем, затем все участники используют тот же SHA. Этот блок также не обучает Food-101.

In [ ]:
# Выполнить один раз координатору; передать model_revision.json участникам.
from configs import MODEL_ID
revision_file = PERSISTENT_ROOT / 'protocol' / 'model_revision.json'
if revision_file.exists():
    model_revision = json.loads(revision_file.read_text())['revision']
else:
    model_revision = subprocess.check_output([PYTHON, '-c', "from huggingface_hub import HfApi; print(HfApi().model_info('google/vit-base-patch16-224-in21k').sha)"], text=True).strip()
    revision_file.parent.mkdir(parents=True, exist_ok=True)
    revision_file.write_text(json.dumps({'model': MODEL_ID, 'revision': model_revision}, indent=2))
subprocess.run([PYTHON, 'test_params.py', '--config', 'B-iso', '--device', 'cuda',
                '--image-size', '224', '--pretrained', '--revision', model_revision,
                '--output', str(out / 'pretrained_smoke.json')], check=True)


## Сохранить свидетельство окружения
Список ниже — фактически установленные версии, ещё не окончательный общий lock. Перед завершением сессии убедитесь, что отчёты появились на Drive. Для source/модели и полной серии сохраняются отдельные hashes/revisions.

In [ ]:
result = subprocess.run([sys.executable, '-m', 'pip', '--python', PYTHON, 'freeze'], capture_output=True, text=True, check=True)
(out / 'pip-freeze.txt').write_text(result.stdout)
(out / 'SOURCE_MANIFEST.json').write_text(json.dumps(manifest, indent=2))
print('Технические отчёты:', out)


## Шаг 3: подготовка manifest Food-101 (отдельный запуск)
Выполняет один координатор после проверки окружения. Не LR search и не test.
Нужно скачать train/cache и проверить 20 200 выбранных изображений. Не запускайте параллельно у пяти участников.

In [ ]:
PREPARE_DATA = False  # Включить один раз координатору.
if PREPARE_DATA:
    import data
    data_manifest = PERSISTENT_ROOT / 'protocol' / 'data_manifest.json'
    if data_manifest.exists():
        subprocess.run([PYTHON, 'data.py', '--verify', '--manifest', str(data_manifest)], check=True)
    else:
        data_sha_path = PERSISTENT_ROOT / 'protocol' / 'dataset_revision.json'
        if data_sha_path.exists():
            data_sha = json.loads(data_sha_path.read_text())['revision']
        else:
            data_sha = subprocess.check_output([PYTHON, '-c', "from huggingface_hub import HfApi; print(HfApi().dataset_info('ethz/food101').sha)"], text=True).strip()
            data_sha_path.parent.mkdir(parents=True, exist_ok=True)
            data_sha_path.write_text(json.dumps({'dataset': data.DATASET_ID, 'revision': data_sha}, indent=2))
        subprocess.run([PYTHON, 'data.py', '--dataset-revision', data_sha, '--manifest', str(data_manifest)], check=True)
